# リハビリ離脱率分析ポートフォリオ

理学療法士としての臨床経験をもとに、整形外科外来リハビリにおける**離脱率**に影響する要因をSQL・統計分析で明らかにする。

※本データは架空データです。実患者の個人情報は一切含まれていません。

## 1. セットアップ：CSV → SQLite変換

In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

# 日本語フォント設定（Windowsの場合は'MS Gothic'、Macは'Hiragino Sans'）
plt.rcParams['font.family'] = 'Hiragino Sans'  # Macの場合


sns.set_style('whitegrid')
%matplotlib inline

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# CSVを読み込んでSQLiteデータベースに変換
patients   = pd.read_csv('patients.csv')
clinical   = pd.read_csv('clinical_records.csv')
diseases   = pd.read_csv('master_diseases.csv')
therapists = pd.read_csv('master_therapists.csv')

conn = sqlite3.connect('rehab_portfolio.db')

patients.to_sql('patients', conn, if_exists='replace', index=False)
clinical.to_sql('clinical_records', conn, if_exists='replace', index=False)
diseases.to_sql('master_diseases', conn, if_exists='replace', index=False)
therapists.to_sql('master_therapists', conn, if_exists='replace', index=False)

print('SQLiteデータベース作成完了: rehab_portfolio.db')
print(f'  patients          : {len(patients)}行')
print(f'  clinical_records  : {len(clinical)}行')
print(f'  master_diseases   : {len(diseases)}行')
print(f'  master_therapists : {len(therapists)}行')

In [2]:
# ipython-sqlを使えるようにする（pip install ipython-sql が必要）
%load_ext sql
%sql sqlite:///rehab_portfolio.db
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

## 2. SQL分析：基本集計

### 2-1. 全体の離脱率

In [ ]:
%%sql
SELECT
    COUNT(*) AS 総患者数,
    SUM(離脱フラグ) AS 離脱数,
    ROUND(AVG(離脱フラグ) * 100, 1) AS 離脱率
FROM clinical_records;

### 2-2. 疾患別の離脱率（4テーブルJOIN）

In [ ]:
%%sql
SELECT
    d.疾患名,
    COUNT(*) AS 患者数,
    ROUND(AVG(p.年齢), 1) AS 平均年齢,
    ROUND(AVG(c.離脱フラグ) * 100, 1) AS 離脱率
FROM patients p
JOIN master_diseases d   ON p.疾患ID = d.疾患ID
JOIN clinical_records c  ON p.患者ID = c.患者ID
GROUP BY d.疾患名
ORDER BY 離脱率 DESC;

### 2-3. 保険種別 × 離脱率

In [ ]:
%%sql
SELECT
    p.保険種別,
    COUNT(*) AS 患者数,
    ROUND(AVG(p.年齢), 1) AS 平均年齢,
    ROUND(AVG(c.離脱フラグ) * 100, 1) AS 離脱率
FROM patients p
JOIN clinical_records c ON p.患者ID = c.患者ID
GROUP BY p.保険種別
ORDER BY 離脱率 DESC;

### 2-4. 距離区分別 離脱率（CASE WHEN）

In [ ]:
%%sql
SELECT
    CASE
        WHEN 自宅からの距離_km < 2  THEN '①2km未満'
        WHEN 自宅からの距離_km < 5  THEN '②2-5km'
        WHEN 自宅からの距離_km < 10 THEN '③5-10km'
        ELSE '④10km以上'
    END AS 距離区分,
    COUNT(*) AS 患者数,
    ROUND(AVG(離脱フラグ) * 100, 1) AS 離脱率
FROM clinical_records
GROUP BY 距離区分
ORDER BY 距離区分;

### 2-5. 就労状況 × 疾患（梨状筋・筋筋膜性に絞る／サブクエリ）

In [ ]:
%%sql
SELECT
    d.疾患名,
    p.就労状況,
    COUNT(*) AS 患者数,
    ROUND(AVG(c.離脱フラグ) * 100, 1) AS 離脱率
FROM patients p
JOIN master_diseases d  ON p.疾患ID = d.疾患ID
JOIN clinical_records c ON p.患者ID = c.患者ID
WHERE d.疾患名 IN ('梨状筋症候群', '筋筋膜性腰痛症')
GROUP BY d.疾患名, p.就労状況
ORDER BY d.疾患名, 離脱率 DESC;

## 3. Python分析：可視化

SQL結果をDataFrameとして取得し、グラフ化する

In [ ]:
# 分析用に全テーブルを結合
query = '''
SELECT p.*, c.初回FIMスコア, c.初回疼痛VAS, c.自宅からの距離_km,
       c.週治療頻度, c.治療期間_週, c.離脱フラグ,
       d.疾患名, d.部位, t.氏名 AS 担当セラピスト, t.経験年数
FROM patients p
JOIN clinical_records c  ON p.患者ID = c.患者ID
JOIN master_diseases d   ON p.疾患ID = d.疾患ID
JOIN master_therapists t ON p.セラピストID = t.セラピストID
'''
df = pd.read_sql(query, conn)
df.head()

### 3-1. 疾患別 離脱率（横棒グラフ）

In [ ]:
disease_dropout = df.groupby('疾患名')['離脱フラグ'].mean().sort_values() * 100

plt.figure(figsize=(8, 6))
colors = ['#d62728' if v >= 40 else '#1f77b4' for v in disease_dropout.values]
plt.barh(disease_dropout.index, disease_dropout.values, color=colors)
plt.xlabel('離脱率 (%)')
plt.title('疾患別 離脱率')
plt.tight_layout()
plt.show()

### 3-2. VASスコアと離脱率の関係（箱ひげ図）

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x='離脱フラグ', y='初回疼痛VAS', palette=['#1f77b4', '#d62728'])
plt.xticks([0, 1], ['継続', '離脱'])
plt.xlabel('')
plt.ylabel('初回疼痛VAS')
plt.title('離脱有無別 初回VASスコアの分布')
plt.tight_layout()
plt.show()

### 3-3. 距離と離脱率の関係

In [ ]:
df['距離区分'] = pd.cut(df['自宅からの距離_km'], bins=[0,2,5,10,30],
                      labels=['〜2km','2〜5km','5〜10km','10km〜'])
dist_dropout = df.groupby('距離区分', observed=True)['離脱フラグ'].mean() * 100

plt.figure(figsize=(7, 5))
plt.bar(dist_dropout.index.astype(str), dist_dropout.values, color='#2ca02c')
plt.ylabel('離脱率 (%)')
plt.title('通院距離別 離脱率')
plt.tight_layout()
plt.show()

## 4. 統計検定

「見えた傾向が統計的に意味があるか」を検証する

### 4-1. カイ二乗検定：疾患名 × 離脱フラグ

In [ ]:
cross_tab = pd.crosstab(df['疾患名'], df['離脱フラグ'])
chi2, p, dof, expected = stats.chi2_contingency(cross_tab)

print(f'カイ二乗統計量: {chi2:.2f}')
print(f'p値          : {p:.5f}')
print(f'判定         : {"有意差あり（疾患と離脱は関連あり）" if p < 0.05 else "有意差なし"}')

### 4-2. t検定：離脱群 vs 継続群の距離

In [ ]:
group_continue = df[df['離脱フラグ']==0]['自宅からの距離_km']
group_dropout  = df[df['離脱フラグ']==1]['自宅からの距離_km']

t_stat, p_val = stats.ttest_ind(group_continue, group_dropout)

print(f'継続群 平均距離: {group_continue.mean():.2f} km')
print(f'離脱群 平均距離: {group_dropout.mean():.2f} km')
print(f't値   : {t_stat:.3f}')
print(f'p値   : {p_val:.5f}')
print(f'判定  : {"有意差あり" if p_val < 0.05 else "有意差なし"}')

### 4-3. ロジスティック回帰：離脱に最も影響する要因は？

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 説明変数を準備（カテゴリ変数はダミー変数化）
features = df[['年齢','初回疼痛VAS','自宅からの距離_km','週治療頻度',
               '独居フラグ','併存疾患あり','初回FIMスコア']].copy()
features['就労中フラグ'] = (df['就労状況']=='就労中').astype(int)

X = features
y = df['離脱フラグ']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression()
model.fit(X_scaled, y)

# オッズ比で影響度を確認
odds_ratio = pd.DataFrame({
    '変数': X.columns,
    '係数': model.coef_[0],
    'オッズ比': np.exp(model.coef_[0])
}).sort_values('オッズ比', ascending=False)

odds_ratio

In [ ]:
plt.figure(figsize=(7, 5))
colors = ['#d62728' if v > 1 else '#1f77b4' for v in odds_ratio['オッズ比']]
plt.barh(odds_ratio['変数'], odds_ratio['オッズ比'], color=colors)
plt.axvline(x=1, color='gray', linestyle='--')
plt.xlabel('オッズ比')
plt.title('離脱リスクに対する各変数のオッズ比')
plt.tight_layout()
plt.show()

## 5. 分析結果のまとめ・提言

（分析結果を見ながらここに記入していく）

### 主な発見
- 

### 提言
- 